# 04 — Anomaly Detection (Classical Pipeline)
## Classical vs Multi-Agent · Reply × LUISS 2026

**Obiettivo:** Rilevare rotte aeroportuali anomale usando un **ensemble di tre modelli**:

| Modello | Approccio | Vantaggi |
|---------|-----------|---------|
| **IsolationForest** | Isola i punti con pochi split casuali | Robusto su alta dimensionalità, efficiente |
| **Local Outlier Factor** | Densità locale rispetto ai vicini | Trova anomalie locali in cluster |
| **Z-score Ensemble** | Flag statistici dalla baseline | Interpretabile, riferito a soglie esplicite |

I tre modelli vengono **combinati** in un unico `anomaly_score` finale per ogni rotta.

---
**Input:** `data/processed/features_with_baseline.csv` (567 rotte × 87 col)  
**Output:** `data/processed/anomaly_results.csv` — score finale + ranking + label anomalia


In [12]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

ROOT     = Path.cwd().parent if (Path.cwd().parent / "data").exists() else Path.cwd()
PROC_DIR = ROOT / "data" / "processed"

print(f"ROOT: {ROOT}")


ROOT: /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent


## 1. Caricamento dati

In [13]:
# Dataset con Z-scores e flag dal notebook 03
features_wb = pd.read_csv(PROC_DIR / "features_with_baseline.csv")

# Metadati baseline e feature cols
with open(PROC_DIR / "baseline_stats.json") as f:
    baseline_meta = json.load(f)
with open(PROC_DIR / "feature_cols.json") as f:
    feat_meta = json.load(f)

ANOMALY_FEATURES = baseline_meta["anomaly_features"]

print(f"features_with_baseline: {features_wb.shape[0]} rotte × {features_wb.shape[1]} colonne")
print(f"Feature per anomaly detection: {len(ANOMALY_FEATURES)}")
print(f"  {ANOMALY_FEATURES}")


features_with_baseline: 567 rotte × 87 colonne
Feature per anomaly detection: 11
  ['tot_allarmi_log', 'pct_interpol', 'pct_sdi', 'pct_nsis', 'tasso_chiusura', 'tasso_rilevanza', 'tasso_allarme_medio', 'tasso_inv_medio', 'score_rischio_esiti', 'tasso_respinti', 'tasso_fermati']


## 2. Preparazione matrice feature

Standardizziamo le feature con `StandardScaler` (media 0, std 1) prima di passarle ai modelli.
IsolationForest e LOF sono sensibili alla scala delle feature.


In [14]:
X_raw = features_wb[ANOMALY_FEATURES].copy()

# StandardScaler: media 0, std 1 per ogni feature
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=ANOMALY_FEATURES, index=features_wb.index)

print(f"Feature matrix X: {X_scaled.shape}")
print(f"Media dopo scaling (deve essere ~0): {X_scaled.mean(axis=0).round(4).tolist()[:3]}...")
print(f"Std dopo scaling (deve essere ~1):   {X_scaled.std(axis=0).round(4).tolist()[:3]}...")


Feature matrix X: (567, 11)
Media dopo scaling (deve essere ~0): [0.0, 0.0, 0.0]...
Std dopo scaling (deve essere ~1):   [1.0, 1.0, 1.0]...


## 3. Modello 1: Isolation Forest

**Principio:** Costruisce alberi decisionali casuali. Le anomalie vengono *isolate*
in meno split rispetto ai punti normali.

**Score:** `anomaly_score_if` ∈ [-1, 1]. Valori vicini a -1 indicano anomalia.  
**Parametri chiave:**
- `contamination=0.1` — assume che ~10% delle rotte siano potenzialmente anomale
- `n_estimators=200` — foresta con 200 alberi per stabilità
- `random_state=42` — riproducibilità


In [15]:
# Isolation Forest
iso_forest = IsolationForest(
    n_estimators   = 200,
    contamination  = 0.10,    # ~10% anomalie attese
    max_samples    = "auto",
    random_state   = 42,
    n_jobs         = -1,
)
iso_forest.fit(X_scaled)

# score_samples: più negativo = più anomalo
# decision_function: < 0 = anomalia, > 0 = normale
if_scores    = iso_forest.score_samples(X_scaled)    # raw scores
if_labels    = iso_forest.predict(X_scaled)           # -1 = anomalia, +1 = normale
if_decisions = iso_forest.decision_function(X_scaled) # < 0 → anomalia

# Normalizza score a [0,1] dove 1 = anomalia
# Invertiamo: score_samples più negativo → anomaly_score più alto
if_anomaly = (if_scores - if_scores.max()) / (if_scores.min() - if_scores.max())
if_anomaly = np.clip(if_anomaly, 0, 1)

n_if_anomalie = (if_labels == -1).sum()
print(f"IsolationForest:")
print(f"  Anomalie rilevate: {n_if_anomalie} ({n_if_anomalie/len(features_wb)*100:.1f}%)")
print(f"  anomaly_score_if: min={if_anomaly.min():.4f}, max={if_anomaly.max():.4f}, mean={if_anomaly.mean():.4f}")


IsolationForest:
  Anomalie rilevate: 57 (10.1%)
  anomaly_score_if: min=-0.0000, max=1.0000, mean=0.3014


## 4. Modello 2: Local Outlier Factor (LOF)

**Principio:** Confronta la densità locale di ogni punto con quella dei suoi vicini.
Un punto è anomalo se la sua densità è molto inferiore a quella dei vicini.

**Parametri chiave:**
- `n_neighbors=20` — considera i 20 vicini più prossimi
- `contamination=0.10` — soglia per classificazione binaria
- `novelty=False` — modalità inductive (si adatta ai dati stessi)


In [16]:
# Local Outlier Factor
lof = LocalOutlierFactor(
    n_neighbors   = 20,
    contamination = 0.10,
    metric        = "euclidean",
    n_jobs        = -1,
)
lof_labels = lof.fit_predict(X_scaled)   # -1 = anomalia, +1 = normale

# negative_outlier_factor_: più negativo = più anomalo
lof_scores = lof.negative_outlier_factor_

# Normalizza a [0,1] dove 1 = anomalia
lof_anomaly = (lof_scores - lof_scores.max()) / (lof_scores.min() - lof_scores.max())
lof_anomaly = np.clip(lof_anomaly, 0, 1)

n_lof_anomalie = (lof_labels == -1).sum()
print(f"Local Outlier Factor:")
print(f"  Anomalie rilevate: {n_lof_anomalie} ({n_lof_anomalie/len(features_wb)*100:.1f}%)")
print(f"  anomaly_score_lof: min={lof_anomaly.min():.4f}, max={lof_anomaly.max():.4f}, mean={lof_anomaly.mean():.4f}")


Local Outlier Factor:
  Anomalie rilevate: 57 (10.1%)
  anomaly_score_lof: min=-0.0000, max=1.0000, mean=0.0238


## 5. Modello 3: Z-score Ensemble (dalla baseline)

Il terzo segnale viene direttamente dalla baseline costruita nel notebook 03:
- `pct_anomalie` = frazione di feature statisticamente anomale per rotta (già calcolato)
- Normalizzato a [0,1] dove 1 = tutte le feature sono fuori soglia


In [17]:
# Z-score ensemble: già calcolato in 03_baseline_construction
# pct_anomalie ∈ [0,1], dove 1 = tutte le 11 feature sono anomale
z_anomaly = features_wb["pct_anomalie"].values

print(f"Z-score Ensemble (baseline):")
print(f"  Rotte con pct_anomalie > 0:    {(z_anomaly > 0).sum()}")
print(f"  Rotte con pct_anomalie >= 0.27: {(z_anomaly >= 0.27).sum()}")
print(f"  z_anomaly: min={z_anomaly.min():.4f}, max={z_anomaly.max():.4f}, mean={z_anomaly.mean():.4f}")


Z-score Ensemble (baseline):
  Rotte con pct_anomalie > 0:    229
  Rotte con pct_anomalie >= 0.27: 11
  z_anomaly: min=0.0000, max=0.3636, mean=0.0473


## 6. Ensemble dei tre modelli

Combiniamo i tre score con **pesi bilanciati**:

| Modello | Peso | Razionale |
|---------|------|-----------|
| IsolationForest | 40% | Modello ML principale, gestisce alta dimensionalità |
| LOF | 35% | Cattura anomalie locali non viste da IF |
| Z-score | 25% | Segnale statistico interpretabile dalla baseline |

**Score finale** `anomaly_score` ∈ [0, 1]:
- ≥ 0.45 → **ALTA ANOMALIA** 
- 0.30–0.45 → **MEDIA ANOMALIA** 
- < 0.30 → **NORMALE**


In [18]:
# Pesi ensemble
W_IF  = 0.40
W_LOF = 0.35
W_Z   = 0.25

anomaly_score = (
    W_IF  * if_anomaly  +
    W_LOF * lof_anomaly +
    W_Z   * z_anomaly
)
anomaly_score = np.clip(anomaly_score, 0, 1)

# ── SOGLIE CALIBRATE ────────────────────────────────────────────────────────
# IF e LOF trovano anomalie diverse (overlap basso) → il score combinato
# raggiunge max ~0.51. Soglie calibrate per questo dataset:
# ALTA   ≥ 0.45 → anomalia confermata da almeno 2 segnali forti
# MEDIA  ≥ 0.30 → anomalia segnalata da almeno 1 modello
# NORMALE < 0.30
def classify_anomaly(score):
    if score >= 0.45:   return "ALTA"
    elif score >= 0.30: return "MEDIA"
    else:               return "NORMALE"

anomaly_label = np.array([classify_anomaly(s) for s in anomaly_score])

print("Ensemble anomaly_score:")
print(f"  min={anomaly_score.min():.4f}, max={anomaly_score.max():.4f}, mean={anomaly_score.mean():.4f}")
print(f"\nDistribuzione label (soglie: ALTA≥0.45, MEDIA≥0.30):")
unique, counts = np.unique(anomaly_label, return_counts=True)
for label, cnt in zip(unique, counts):
    print(f"  {label:<10}: {cnt:>4} rotte ({cnt/len(features_wb)*100:.1f}%)")


Ensemble anomaly_score:
  min=0.0000, max=0.5128, mean=0.1407

Distribuzione label (soglie: ALTA≥0.45, MEDIA≥0.30):
  ALTA      :    3 rotte (0.5%)
  MEDIA     :   24 rotte (4.2%)
  NORMALE   :  540 rotte (95.2%)


## 7. Risultati e ranking

In [19]:
# Costruisce dataframe risultati
results = features_wb[["ROTTA", "PAESE_PART", "ZONA", "score_composito",
                        "n_anomalie", "tot_allarmi_log", "pct_interpol",
                        "score_rischio_esiti"]].copy()

results["anomaly_score_if"]  = if_anomaly.round(4)
results["anomaly_score_lof"] = lof_anomaly.round(4)
results["anomaly_score_z"]   = z_anomaly.round(4)
results["anomaly_score"]     = anomaly_score.round(4)
results["anomaly_label"]     = anomaly_label
results["if_flag"]           = (if_labels == -1).astype(int)
results["lof_flag"]          = (lof_labels == -1).astype(int)
results["n_models_flagged"]  = results["if_flag"] + results["lof_flag"] + (features_wb["n_anomalie"] >= 3).astype(int)

results = results.sort_values("anomaly_score", ascending=False).reset_index(drop=True)
results["rank"] = results.index + 1

print("Top 20 rotte anomale:")
top20_cols = ["rank","ROTTA","PAESE_PART","anomaly_label","anomaly_score",
              "anomaly_score_if","anomaly_score_lof","anomaly_score_z",
              "n_models_flagged","tot_allarmi_log","pct_interpol"]
print(results[top20_cols].head(20).to_string(index=False))


Top 20 rotte anomale:
 rank   ROTTA     PAESE_PART anomaly_label  anomaly_score  anomaly_score_if  anomaly_score_lof  anomaly_score_z  n_models_flagged  tot_allarmi_log  pct_interpol
    1 GYD-FCO    Azerbaigian          ALTA         0.5128            0.4210             0.9839           0.0000                 1           1.7918        0.2500
    2 ALG-MXP        Algeria          ALTA         0.4682            1.0000             0.0000           0.2727                 3           5.1874        0.2500
    3 KIV-FCO       Moldavia          ALTA         0.4558            0.2077             1.0000           0.0909                 1           5.2364        0.0000
    4 RUH-VCE Arabia Saudita         MEDIA         0.4352            0.8608             0.0000           0.3636                 3           1.6094        0.0000
    5 HRG-NAP         Egitto         MEDIA         0.4322            0.3631             0.7549           0.0909                 1           5.0499        0.0000
    6 CGK-FC

## 8. Analisi overlap modelli

In [20]:
# Quante rotte vengono flaggate da 1, 2, 3 modelli?
print("Concordanza tra modelli:")
print(f"  Flaggate da tutti e 3:       {(results['n_models_flagged'] == 3).sum()}")
print(f"  Flaggate da almeno 2:        {(results['n_models_flagged'] >= 2).sum()}")
print(f"  Flaggate da almeno 1:        {(results['n_models_flagged'] >= 1).sum()}")

# Set overlap
set_if  = set(results[results["if_flag"]  == 1]["ROTTA"])
set_lof = set(results[results["lof_flag"] == 1]["ROTTA"])
set_z   = set(results[features_wb["n_anomalie"] >= 3]["ROTTA"])

print(f"\nOverlap IF ∩ LOF:       {len(set_if & set_lof)}")
print(f"Overlap IF ∩ Z-score:    {len(set_if & set_z)}")
print(f"Overlap LOF ∩ Z-score:   {len(set_lof & set_z)}")
print(f"Overlap IF ∩ LOF ∩ Z:    {len(set_if & set_lof & set_z)}")
print(f"\nRotte nel consenso (tutti e 3 modelli):")
consensus = set_if & set_lof & set_z
for r in sorted(consensus):
    row = results[results["ROTTA"] == r].iloc[0]
    print(f"  {r:<12} score={row['anomaly_score']:.4f}  paese={row['PAESE_PART']}")


Concordanza tra modelli:
  Flaggate da tutti e 3:       2
  Flaggate da almeno 2:        6
  Flaggate da almeno 1:        117

Overlap IF ∩ LOF:       4
Overlap IF ∩ Z-score:    1
Overlap LOF ∩ Z-score:   2
Overlap IF ∩ LOF ∩ Z:    0

Rotte nel consenso (tutti e 3 modelli):


## 9. Validazione e quality check

In [21]:
sep = "=" * 64
print(sep)
print("  QUALITY CHECK — anomaly_results.csv")
print(sep)
print(f"\n  Rotte totali:          {len(results)}")
print(f"  Null in results:       {results.isnull().sum().sum()}")
print(f"  Score range:           [{results['anomaly_score'].min():.4f}, {results['anomaly_score'].max():.4f}]")
print(f"\n  Label distribution:")
for lab in ["ALTA","MEDIA","NORMALE"]:
    n = (results["anomaly_label"] == lab).sum()
    print(f"    {lab:<10}: {n:>4} ({n/len(results)*100:.1f}%)")
print(f"\n  Top 5 rotte con massimo rischio:")
for _, row in results.head(5).iterrows():
    print(f"    [{row['rank']:>3}] {row['ROTTA']:<12} {row['anomaly_label']:<8} score={row['anomaly_score']:.4f}  paese={row['PAESE_PART']}")
print(sep)


  QUALITY CHECK — anomaly_results.csv

  Rotte totali:          567
  Null in results:       0
  Score range:           [0.0000, 0.5128]

  Label distribution:
    ALTA      :    3 (0.5%)
    MEDIA     :   24 (4.2%)
    NORMALE   :  540 (95.2%)

  Top 5 rotte con massimo rischio:
    [  1] GYD-FCO      ALTA     score=0.5128  paese=Azerbaigian
    [  2] ALG-MXP      ALTA     score=0.4682  paese=Algeria
    [  3] KIV-FCO      ALTA     score=0.4558  paese=Moldavia
    [  4] RUH-VCE      MEDIA    score=0.4352  paese=Arabia Saudita
    [  5] HRG-NAP      MEDIA    score=0.4322  paese=Egitto


## 10. Salvataggio

In [22]:
out_csv = PROC_DIR / "anomaly_results.csv"
results.to_csv(out_csv, index=False)

# Salva anche il summary JSON
summary = {
    "n_routes"      : len(results),
    "n_alta"        : int((results["anomaly_label"] == "ALTA").sum()),
    "n_media"       : int((results["anomaly_label"] == "MEDIA").sum()),
    "n_normale"     : int((results["anomaly_label"] == "NORMALE").sum()),
    "consensus_routes": sorted(list(set_if & set_lof & set_z)),
    "model_weights" : {"isolation_forest": W_IF, "lof": W_LOF, "z_score": W_Z},
    "top10_routes"  : results.head(10)["ROTTA"].tolist(),
}
with open(PROC_DIR / "anomaly_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"✓ anomaly_results.csv  — {len(results)} rotte, {results.shape[1]} colonne")
print(f"✓ anomaly_summary.json — {summary['n_alta']} ALTA, {summary['n_media']} MEDIA, {summary['n_normale']} NORMALE")
print(f"\nPercorso: {out_csv}")


✓ anomaly_results.csv  — 567 rotte, 17 colonne
✓ anomaly_summary.json — 3 ALTA, 24 MEDIA, 540 NORMALE

Percorso: /Users/danielegiovanardi/classical-vs-multiagent/classical-vs-multiagent/data/processed/anomaly_results.csv


---
## Riepilogo

| Modello | Metodo | Anomalie (~10%) |
|---------|--------|----------------|
| IsolationForest | Isolamento random | ~57 rotte |
| LOF | Densità locale | ~57 rotte |
| Z-score Ensemble | Soglie statistiche | 11 rotte (≥3 feature) |
| **Ensemble combinato** | Pesato 40/35/25 | label ALTA/MEDIA/NORMALE |

### Prossimo notebook
**`05_post_processing.ipynb`** — Business rules layer:
- Regole di business sovrapposte al ML (es. alert rate > 3× baseline, INTERPOL flag)
- Filtraggio risultati per soglia di confidenza
- Generazione profili di rischio per rotta
